# Phase 3: UMC 분류 — Claude Code 오프라인 워크플로우

이 노트북은 `src/phase03_llm_analysis.py`의 동일 로직을 단계별로 실행합니다.

## 워크플로우

```
1. [prepare] 구별 CSV → 배치 입력 마크다운 생성
2. [수동]    phase03_batches/*.md → Claude Code 분석 → phase03_responses/ 저장
3. [parse]   Claude 응답 → 배치 별 CSV
4. [merge]   배치 CSV + 원본 → 최종 03_umc_classified.csv
```


In [ ]:
# 프로젝트 루트 설정
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"프로젝트 루트: {PROJECT_ROOT}")

## Step 1: Prepare — 배치 입력 마크다운 생성

`data/processed/split_by_gu/{구명}.csv` → `data/processed/phase03_batches/{구명}_batch{N}.md`

In [ ]:
from src.phase03_llm_analysis import run_prepare

# 전체 구 처리
run_prepare(gu_list=None, batch_size=50)

# 특정 구만 처리하려면:
# run_prepare(gu_list=['종로구', '강남구'], batch_size=50)

## ⏸ 수동 단계 (Claude Code)

1. `data/processed/phase03_batches/` 폴더의 배치 파일들을 Claude Code에서 열기
2. `CLAUDE.md`가 프로젝트 루트에 있으므로 Claude Code가 자동으로 시스템 프롬프트로 인식
3. `umc_classification_prompt.md` 기준으로 각 배치 분류
4. 응답을 `data/processed/phase03_responses/{동일 파일명}` 으로 저장

저장 후 아래 parse 단계 진행

## Step 2: Parse — Claude 응답 파싱

`data/processed/phase03_responses/{구명}_batch{N}.md` → `data/processed/phase03_parsed/{구명}_batch{N}.csv`

In [ ]:
from src.phase03_llm_analysis import run_parse

# 전체 응답 파싱
run_parse(gu_list=None)

# 특정 구만:
# run_parse(gu_list=['종로구'])

## Step 3: Merge — 최종 CSV 생성

파싱된 배치 CSV + 원본 필터링 데이터 → `data/processed/03_umc_classified.csv`

In [ ]:
from src.phase03_llm_analysis import run_merge

run_merge(
    source_csv="data/processed/02_keyword_filtered.csv",
    output_csv="data/processed/03_umc_classified.csv",
)

## Step 4: 결과 확인

In [ ]:
import pandas as pd

df = pd.read_csv(PROJECT_ROOT / "data/processed/03_umc_classified.csv", encoding="utf-8")
print(f"전체 분류 건수: {len(df):,}")
print(f"컬럼: {list(df.columns)}")
print()

print("[UMC 관련성 분포]")
print(df["umc_related"].value_counts())
print()

print("[UMC 차원 빈도 (상위 10)]")
# 쉼표 구분 다중 차원 처리
dim_series = df["umc_dimensions"].dropna().str.split(",").explode().str.strip()
print(dim_series.value_counts().head(10))

In [ ]:
# 샘플 출력 (UMC 관련 Y인 것)
umc_y = df[df["umc_related"] == "Y"][["dbId", "gu", "title", "umc_dimensions", "problem_group"]]
umc_y.head(10)